In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 20.4 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
# 1. Dataset
documents = [
    "FAISS is a library for efficient similarity search of dense vectors.",
    "BM25 is a ranking function used in keyword-based search engines.",
    "Semantic search uses embeddings to find meaning-based matches.",
    "Hybrid search combines keyword search with vector similarity.",
    "Deep learning uses neural networks for representation learning."
]
# 2. Embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
# 3. Vector search (FAISS)
embeddings = model.encode(documents, convert_to_numpy=True).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
# 4. BM25 (keyword search)
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)
# 5. Vector retrieval
def vector_search(query, top_k=3):
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_emb, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append((documents[idx], float(distances[0][i])))
    return results
# 6. BM25 retrieval
def bm25_search(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    ranked = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked:
        results.append((documents[idx], float(scores[idx])))
    return results
# 7. Hybrid retrieval
def hybrid_search(query, top_k=3):
    vec_results = vector_search(query, top_k=top_k)
    bm25_results = bm25_search(query, top_k=top_k)

    combined = {}

    # Add vector results
    for text, score in vec_results:
        combined[text] = combined.get(text, 0) + (1 / (1 + score))  # convert distance to similarity

    # Add BM25 results
    for text, score in bm25_results:
        combined[text] = combined.get(text, 0) + score

    # Sort
    sorted_results = sorted(combined.items(), key=lambda x: x[1], reverse=True)

    return sorted_results[:top_k]

# 8. Simple answer generator
# (mock LLM: just returns top passage)
def generate_answer(results):
    return results[0][0]

# 9. Compare both
query = "What is hybrid search?"

print("QUERY:", query)
print("\n--- VECTOR SEARCH ---")
vec_results = vector_search(query)

for text, score in vec_results:
    print(f"Score: {score:.4f}")
    print("Text:", text)
    print()

vec_answer = generate_answer(vec_results)
print("Final Answer (Vector):", vec_answer)

print("\n--- HYBRID SEARCH ---")
hyb_results = hybrid_search(query)

for text, score in hyb_results:
    print(f"Score: {score:.4f}")
    print("Text:", text)
    print()

hyb_answer = generate_answer(hyb_results)
print("Final Answer (Hybrid):", hyb_answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY: What is hybrid search?

--- VECTOR SEARCH ---
Score: 0.6988
Text: Hybrid search combines keyword search with vector similarity.

Score: 1.1408
Text: Semantic search uses embeddings to find meaning-based matches.

Score: 1.2452
Text: BM25 is a ranking function used in keyword-based search engines.

Final Answer (Vector): Hybrid search combines keyword search with vector similarity.

--- HYBRID SEARCH ---
Score: 1.7451
Text: Hybrid search combines keyword search with vector similarity.

Score: 0.7658
Text: BM25 is a ranking function used in keyword-based search engines.

Score: 0.4671
Text: Semantic search uses embeddings to find meaning-based matches.

Final Answer (Hybrid): Hybrid search combines keyword search with vector similarity.


In [ ]:
!pip -q install sentence-transformers transformers rank-bm25

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi
import numpy as np
import torch
import re
# Knowledge base
docs = [
    "Python was created by Guido van Rossum.",
    "The chemical formula of water is H2O.",
    "Mount Everest is the highest mountain above sea level.",
    "The human heart has four chambers.",
    "The blue whale is the largest animal on Earth.",
    "The capital of France is Paris.",
    "The Sun is a star at the center of the Solar System.",
    "The Moon is Earth's natural satellite.",
    "The Great Wall is in China.",
    "Bats are mammals, not birds."
]
# Models
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)
# Embeddings
doc_emb = embed_model.encode(
    docs,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# -----------------------
# BM25
# -----------------------
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_docs = [tokenize(doc) for doc in docs]
bm25 = BM25Okapi(tokenized_docs)

# -----------------------
# Hybrid retrieval using RAW scores
# hybrid = alpha*vector_raw + beta*bm25_raw
# -----------------------
def retrieve_hybrid(query, top_k=3, alpha=0.7, beta=0.3):
    q_emb = embed_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    vec_scores = np.dot(doc_emb, q_emb)   # raw vector scores
    bm25_scores = np.array(bm25.get_scores(tokenize(query)), dtype=np.float32)  # raw bm25 scores

    results = []
    for i, doc in enumerate(docs):
        hybrid_score = alpha * float(vec_scores[i]) + beta * float(bm25_scores[i])
        results.append({
            "doc": doc,
            "vector_score": float(vec_scores[i]),
            "bm25_score": float(bm25_scores[i]),
            "hybrid_score": float(hybrid_score)
        })

    results.sort(key=lambda x: x["hybrid_score"], reverse=True)
    return results[:top_k]

# -----------------------
# Answer generation
# -----------------------
def generate_answer(question, passages):
    context = " ".join(passages)
    prompt = f"""Answer using only the context.
If the answer is not in the context, say: I don't know.

Context: {context}
Question: {question}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = llm.generate(**inputs, max_new_tokens=30)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# Run
def ask(question, top_k=3):
    print("=" * 100)
    print("QUESTION:", question)
    print("\n--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---")

    results = retrieve_hybrid(question, top_k=top_k)

    for r in results:
        print(f"Doc: {r['doc']}")
        print(f"Vector Score: {r['vector_score']:.4f}")
        print(f"BM25 Score  : {r['bm25_score']:.4f}")
        print(f"Hybrid Score: {r['hybrid_score']:.4f}\n")

    answer = generate_answer(question, [r["doc"] for r in results])
    print("Final Answer:", answer)
# Test
questions = [
    "Who created Python?",
    "What is the formula of water?",
    "Which animal is the largest on Earth?",
    "What is the highest mountain above sea level?",
    "What is the capital of France?"
]

for q in questions:
    ask(q)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

QUESTION: Who created Python?

--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---
Doc: Python was created by Guido van Rossum.
Vector Score: 0.8621
BM25 Score  : 3.7837
Hybrid Score: 1.7386

Doc: The Great Wall is in China.
Vector Score: 0.1197
BM25 Score  : 0.0000
Hybrid Score: 0.0838

Doc: The blue whale is the largest animal on Earth.
Vector Score: 0.0760
BM25 Score  : 0.0000
Hybrid Score: 0.0532

Final Answer: Guido van Rossum
QUESTION: What is the formula of water?

--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---
Doc: The chemical formula of water is H2O.
Vector Score: 0.7714
BM25 Score  : 5.4420
Hybrid Score: 2.1726

Doc: The capital of France is Paris.
Vector Score: 0.1013
BM25 Score  : 1.7685
Hybrid Score: 0.6015

Doc: The Sun is a star at the center of the Solar System.
Vector Score: 0.0989
BM25 Score  : 1.5472
Hybrid Score: 0.5334

Final Answer: H2O
QUESTION: Which animal is the largest on Earth?

--- HYBRID RETRIEVAL (RAW VECTOR + RAW BM25) ---
Doc: The blue whale is the larg